<div dir="rtl"><h2>05 — שערים צפויים: מידע מוקדם נכנס, שני ערכי <a href="#glossary">λ</a> יוצאים</h2>
<p>EXP-005 משתמש בעונות הפיתוח בלבד. <a href="#glossary">D3</a>, ההרצה המלאה של סולם הניסוי, הושלם. התוצאות בפרק 6 הן טיוטה; ההכרעות ממתינות לעמית ול-QA (בדיקת איכות עצמאית של Claude). המחברת קוראת תוצרים חתומים ואינה מאמנת מודל.</p><p><a href="#glossary">as_of</a> הוא בוקר החיזוי: כל פיצ׳ר משתמש רק במה שקדם לו. λ הוא ממוצע השערים הצפוי לקבוצה, ולא תוצאה מובטחת.</p></div><div dir="rtl"><a href="#glossary">מילון המושגים</a></div>

In [ ]:
import os
import sys
from pathlib import Path

root = Path.cwd()
while not (root / "config/exp005.yaml").exists():
    if root == root.parent:
        raise RuntimeError("Run from the project checkout")
    root = root.parent
os.chdir(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

In [ ]:
import pandas as pd
import plotly.io as pio
from IPython.display import HTML, display

from ligat.viz import goals_notebook as views
from ligat.viz import goals_qa as qa
from ligat.viz import goals_walk as walk
from ligat.viz.presentation import display_table

pio.renderers.default = "plotly_mimetype"
data = views.development_data()


def table(frame, names, **kwargs):
    kwargs.setdefault("show_index", False)
    kwargs.setdefault("unicode_minus", list(frame.select_dtypes(include="number").columns))
    display(display_table(frame.rename_axis(None), names, columns=list(frame.columns), **kwargs))


def show(figure, name):
    figure.show()


display(HTML(walk.input_summary(data)))
display(HTML(walk.provisional_summary(data)))

<div dir="rtl"><h2>0 — ממתין ל-QA ולהכרעת עמית</h2>
<p>כאן יופיעו הפיצ׳רים שנכנסו, הדיוק, ומה עובר ל-M3. טרם נבחר מודל וטרם נכתבה שכבת λ אמיתית. גם כאשר D3 מסתיים, סימוני כלל 6 נשארים להכרעה אנושית.</p>
</div>

<div dir="rtl"><h2>1 — M2 חוזה שערים לכל קבוצה בבוקר המשחק</h2>
<p>λ הוא ממוצע חיובי של שערים צפויים, לא תוצאה מובטחת ולא הסתברות לניצחון. למשל λ=1.6 אומר 1.6 שערים בממוצע במפגשים חוזרים באותם תנאים; משחק יחיד עדיין יכול להסתיים באפס או בארבעה. M3 מקבל שני λ ובונה מטריצת תוצאות; הסימולציה מתרגמת משחקים לטבלה. התרשים מתאר חוזה נתונים, ללא אומדן או רווח סמך.</p>
</div>

In [ ]:
display(HTML(walk.chain_html()))

<div dir="rtl"><h3>1.1 — המודלים שמשתתפים</h3>
<p><b><a href="#glossary">GLM</a> של Poisson:</b> כל פיצ׳ר מכפיל את קצב השערים בגורם קבוע; למשל ביתיות של פי 1.2 היא דוגמה כללית. מעט פרמטרים נותנים מודל יציב שקל להסביר בכדורגל. EXP-002 הוא סרגל GLM.</p>
<p><b><a href="#glossary">XGBoost</a>:</b> הרבה עצי החלטה קטנים; כל עץ מתקן מעט את טעות הקודמים (boosting). הוא לומד קשרים לא ליניאריים ואינטראקציות, כמו שילוב ביתיות וכוח יריבה, אך במעט נתונים עלול לשנן.</p>
<p><b><a href="#glossary">CatBoost</a>:</b> משפחה דומה של עצים עם טיפול מובנה בקטגוריות, למשל שלב הליגה. משמש להשוואה לשיטת העצים האחרת.</p>
<p><b><a href="#glossary">Maher</a> בסגנון Dixon-Coles:</b> לומד כוח התקפה והגנה לכל קבוצה ויתרון בית. מעדכן את הכוחות לפני כל יום משחקים, כך שניצחונות עבר יכולים לשנות את ציפיית השערים הבאה. הנוסחאות והפרטים בפרק 3.</p></div>

<div dir="rtl"><h2>2.1 — כל משחק תורם שתי שורות, עם יעד משלו</h2>
<p>שורה היא קבוצה במשחק. היעד goals_scored הוא מספר השערים באותו משחק, מתוך matches; goals_for של prematch הוא סכום עבר ואינו היעד. הקלט הוא prematch_v2 הקפוא. ספירת השורות היא מפקד מלא של הנתונים, ולכן אין לה רווח סמך.</p>
</div>

In [ ]:
show(walk.season_games_figure(data), "E3a_season_games")
display(HTML(walk.warmup_html(data)))

<div dir="rtl"><h2>2.2 — בכל קיפול, ההיסטוריה קודמת לעונת המבחן</h2>
<p>Walk-forward מאמן על העבר ובודק על העונה הבאה. עונות החימום אינן משמשות ולידציה; יש 15 קיפולים. מודלי העצים ו-GLM מתאמנים לקיפול. Maher מתעדכן לפני כל יום משחקים, וגם בו משתמשים רק בתוצאות שקדמו לתחילת היום. הצבע מתאר תוכנית קבועה, ולא שיעור שדורש רווח סמך.</p>
</div>

In [ ]:
show(walk.folds_figure(data), "E3a_walk_forward")

<div dir="rtl"><h2>2.3 — זמן מחצית קצר מעדיף את הכדורגל העדכני</h2>
<p>דעיכה היא כמה משחק ישן נספר באימון. ציר x הוא כמה עונות אחורה המשחק נמצא, ציר y הוא המשקל שלו, וכל קו הוא זמן מחצית אחר. זמן מחצית h הוא מספר העונות שאחריהן משחק נספר כחצי.</p>

</div>

$$
w(a;h)=2^{-a/h},\qquad a=s_{validation}-1-s_{training},\qquad w=1\text{ ללא דעיכה}.
$$

<div dir="rtl">
<p>
<p>המרחק נמדד מעונת האימון האחרונה, שמשקלה 1; הטווח הוא מעל 0 ועד 1. כשחוזים את 2023/24 עם h=4, העונה 2018/19 נמצאת ארבע עונות לפני עונת האימון האחרונה ולכן משקלה חצי; עם h=1 משקלה 1/16. בלי דעיכה, כל העונות שוות. המשקל אינו מנורמל מחדש. אלו עקומות מתמטיות מדויקות, ללא אי-ודאות מדגמית.</p><p>ב-Maher אותו עיקרון, אך המרחק הוא הפרש הימים חלקי 365.25. הגרף מציג ימים ובעונות.</p></p></div>
<div dir="rtl"><p>המסקנה מהטבלה שאחרי הגרפים: Maher נשען על תוצאות בלבד ולכן צריך להחליט כמה עבר ליישן. ל-XGBoost יש Elo וכושר שמתעדכנים מעצם הגדרתם; לכן ההבדל בין זמני המחצית שלו קטן יותר. זו פרשנות של דפוס בתוצאות, לא שינוי בשיטת האימון.</p></div>

In [ ]:
from ligat.evaluate.goals_experiment import load_goal_experiment

show(views.decay_figure(), "E2a_season_decay")
show(walk.daily_decay_figure(), "E2a_daily_decay")
display(HTML(walk.decay_result_html()))

_, decay_validation = qa.evidence(load_goal_experiment())
table(
    qa.decay_summary(decay_validation),
    {"model": "מודל", "half": "זמן מחצית בעונות", "nll": "NLL ממוצע", "configurations": "תצורות"},
    precision={"nll": 6},
)

<div dir="rtl"><h2>3.1 — <a href="#glossary">NLL</a> מודד כמה המודל הופתע ממה שקרה</h2>
<p>Poisson מתאר מספר שערים שלם Y ≥ 0, עם קצב λ &gt; 0. NLL הוא מינוס הלוגריתם הטבעי של ההסתברות שהמודל נתן לתוצאה שנצפתה. ערך נמוך טוב; הטווח הוא מאפס ומעלה.</p></div>

$$
P(Y=y\mid\lambda)=e^{-\lambda}\lambda^y/y!,\qquad NLL=-\ln P(Y=y\mid\lambda).
$$

<div dir="rtl"><p>למשל λ=1.6 נותן לשער אחד הסתברות של כ-0.323, ולכן −ln(0.323)≈1.130. NLL של 1.41 פירושו הסתברות גאומטרית של כ-24.4% למה שקרה; זה ממוצע של הסתברויות בקנה מידה לוגריתמי, ולא אחוז ניחושים נכונים. בכדורגל: מודל טוב נותן יותר משקל לתוצאות שבאמת מתרחשות.</p><p>שוויון ממוצע ושונות נבדק במחברת 02, ולא נמצא פיזור יתר. פרק 7 יבדוק כיול מחדש על המודל. Γ ו-deviance מוסברים במילון. הדוגמה הבאה מתארת משחק אמיתי אך λ=1.6 הוא מספר המחשה בלבד.</p></div>

In [ ]:
from scipy.stats import poisson

figure, example = views.poisson_example(data)
example["game_id"] = example.game_id.astype(str)
table(
    example,
    {
        "game_id": "מזהה משחק",
        "team": "קבוצה",
        "opponent": "יריבה",
        "season": "עונה",
        "observed": "שערים בפועל",
        "lambda_example": "λ להמחשה",
        "nll_example": "NLL להמחשה",
    },
    precision={"nll_example": 4},
)
show(figure, "E2a_poisson_example")

poisson_rows = pd.DataFrame(
    {
        "goals": range(5),
        "probability": poisson.pmf(range(5), 1.6),
        "nll": -poisson.logpmf(range(5), 1.6),
    }
)
table(poisson_rows, {"goals": "שערים", "probability": "הסתברות", "nll": "הפתעה NLL"})

<div dir="rtl"><h2>3.2 — XGBoost מוסיף עצים שמתקנים את חיזוי השערים</h2>
</div>

$$
\lambda(x)=\exp\left[b+\sum_{t=1}^{T}\eta f_t(x)\right].
$$

<div dir="rtl">
<p>כל עץ לומד תיקון קטן משילוב פיצ׳רים, למשל פער <a href="#glossary">Elo</a> וביתיות. האקספוננט מבטיח λ חיובי. count:poisson הוא יעד האימון, hist היא שיטת הבנייה. CatBoost משתמש גם הוא בעצים ובהפסד Poisson, עם טיפול משותף בקטגוריות. stage מקודד כן/לא ברשת; מזהי קבוצה ויריבה נבדקים בנפרד בסוף הסולם, עם ערך ״לא מוכרת״ שנקבע מההיסטוריה בלבד.</p>
</div>
<div dir="rtl"><p>דוגמת עץ בעומק 2: קודם ״פער Elo מעל 100?״, אחר כך ״בבית?״. מתקבלים ארבעה קצוות של מצבים כדורגלניים. כל עץ מתקן מעט את לוג קצב השערים. η הוא גודל התיקון ומספר העצים הוא מספר ההזדמנויות לתקן: ביחד זה תקציב למידה. למשל תיקון לוג של 0.1 בקצב 0.05 מוסיף 0.005 ללוג λ, כלומר מכפיל את הציפייה בערך ב-1.005. המונחים count:poisson ו-hist מוסברים בפרק 8.</p></div>

<div dir="rtl"><h2>3.3 — GLM נותן בסיס פשוט, Maher מפריד התקפה והגנה</h2>
</div>

$$
\lambda_i=\exp(\beta_0+x_i^T\beta),\qquad \lambda_{ij}=\alpha_i\delta_j\gamma^{I(home)}.
$$

<div dir="rtl">
<p>ב-GLM מקדם חיובי מגדיל את קצב השערים, למשל יתרון בית קבוע באותו מערך פיצ׳רים. ב-Maher α היא חוזק ההתקפה, δ היא נטייה לספוג, ו-γ היא מכפלת הביתיות; כולם חיוביים. נדרשת מגבלת זיהוי </p></div>

$$
\sum\log\alpha=0
$$

<details><summary>פרטי ה-prior לעולה — הרחבה מתמטית</summary>

<div dir="rtl">. כאן Dixon–Coles הוא קו בסיס Maher עם דעיכה ועדכון יומי; אין תיקון תלויות לתוצאות נמוכות (ρ).
<p>לעולה יש prior קפוא בעוצמה של שישה משחקים, סביב ממוצע ליגת העונה <i>R</i>:
</div>

$$
\frac{k}{2}\sum_{p\in P}\left[(\log\alpha_p-\overline{\log\alpha}_R-a^*)^2+(\log\delta_p-\overline{\log\delta}_R-b^*)^2\right].
$$

<div dir="rtl">
<p>עוצמת הקנס בפועל היא </p></div>

$$
k=6\bar y_w w_0
$$

<div dir="rtl">: ממוצע שערי העבר המשוקלל מתרגם את ששת המשחקים לעוצמה, ו-<i>w₀</i> הוא משקל תחילת העונה ביחס לבוקר החיזוי (1 ללא דעיכה). כך גם המידע המוקדם מתיישן עם העונה.<p>כשיש כמה עולות, נגזרת ממוצע R קושרת את הקנסות. עולה ללא משחקים אינה חייבת להיות בדיוק a*/b*; תנאי האופטימום הוא </p></div>

$$
d_p=\sum_{q\in P}d_q/|R|
$$

<div dir="rtl">. הקנס אינו נכלל ב-NLL הוולידציה.
</div>

</details>

<div dir="rtl"><p>למשל כוח התקפה פי 1.1 ונטיית יריבה לספוג פי 1.2 מכפילים את קצב השערים פי 1.32 לפני הביתיות. הסימון δ לספיגה נפרד ממקדמי β של ה-GLM. ρ אינו כאן: הוא מתקן צירופי תוצאות של שתי קבוצות, עבודה של M3 שתיבדק במחברת 06. כאן חוזים את מספר שערי כל קבוצה בנפרד.</p></div>
<div dir="rtl"><p>ה-prior של עולה קטן מ-1 בהתקפה וגדול מ-1 בספיגה, כפי שמחושב בטבלה. זה מונע הנחה שקבוצה שעולה לליגה כבר חזקה כממוצע הליגה. המקור הוא עונות העולות הקפואות, עם עוצמה שקולה לשישה משחקים.</p></div>

In [ ]:
prior = views.frozen_prior()
table(
    pd.DataFrame(
        [
            dict(
                attack=__import__("math").exp(prior["a_star"]),
                concede=__import__("math").exp(prior["b_star"]),
            )
        ]
    ),
    {"attack": "מכפיל התקפה של עולה", "concede": "מכפיל נטייה לספוג"},
)
table(
    pd.DataFrame([dict(team_seasons=prior["team_seasons"], strength=6)]),
    {"team_seasons": "עונות קבוצה של עולות במקור", "strength": "שווי prior במשחקים"},
)

<div dir="rtl"><h2>4 — הרשת קבועה מראש: דיוק מול מורכבות ורגולריזציה</h2>
<p>היפר-פרמטר הוא בחירה שנקבעת מחוץ ללמידת המקדמים: כמה עמוק העץ, כמה תיקונים וכמה קנס. הרשת כוללת ערכים פשוטים ומורכבים, והבחירה לפי ולידציה עתידית. הטבלה נגזרת ישירות מה-YAML. ערך None בזמן מחצית פירושו בלי דעיכה. ברירת מחדל של ספרייה היא מועמד עצמאי מלא. הגרפים שישוו צירופים יתווספו ב-E2c; ציוני המועמדים הם טיוטה לפני הכרעות עמית.</p>
</div>
<div dir="rtl"><p>שלב A בודק עומק, זהירות, עצים וקצב; שלב B מקבע את הצעת A ובודק דגימה, קנס ודעיכה. ברירת המחדל היא תצורה נפרדת מלאה, ולא ערבוב של ערכים. לדוגמה: עומק 2, 150 עצים וקצב 0.05 מתארים הרבה תיקונים קטנים של מצבים פשוטים. הנימוק הכללי הוא תקציב שמאפשר השוואה בשלוש ספריות בלי חיפוש לא מוגבל; הנימוקים לפרמטרים מפורטים ב-prereg §7.</p></div>
<div dir="rtl"><p>תקציב A הוא 5 עומקים × 3 רמות זהירות × 3 מספרי עצים × 2 קצבים, ועוד ברירת מחדל מלאה. ב-B: 3 קנסות × 2 שיעורי דגימה × 4 זמני מחצית. ברירת המחדל של הספרייה שונה מן הרשת; אסור לייחס נקודה בודדת שלה ל״אפקט״ של פרמטר. השילובים והערכים עצמם נקראים מהקונפיגורציה המאושרת.</p></div>

In [ ]:
display(HTML(qa.grid_explanation(load_goal_experiment())))
hyper = qa.hyperparameter_guide()
table(
    hyper,
    {
        "model": "מודל ושלב",
        "parameter": "פרמטר",
        "values": "ערכים",
        "meaning": "מה הפרמטר עושה",
        "football": "משמעות כדורגלנית",
        "reason": "נימוק במסגרת §7",
    },
)

<div dir="rtl"><h2>4.1 — loss האימון מספר איך המודל לומד את העבר</h2><p>גמישות גדולה יותר מאפשרת להתאים טוב יותר את האימון: באופטימום מדויק של מחלקות מודלים מקוננות וללא שינוי קנס, loss האימון המינימלי אינו עולה. בפועל, דגימה אקראית, רגולריזציה, קצב למידה ואופטימיזציה סופית אינם מבטיחים ירידה בכל השוואת צירופים. רק ולידציה על עתיד עונה אם המורכבות עוזרת לחיזוי.</p><p>הטבלה נגזרת מיומני אימון חתומים, לא מאימון חוזר. train_loss נשמר float32. ב-Maher יש שורה לכל אימון יומי עם איטרציות, loss סופי והתכנסות; העקומה המקורית נשמרת ביומן הדחוס. GLM שהתכנס באפס איטרציות מופיע עם loss חסר, ולא אפס מומצא.</p></div>

In [ ]:
from ligat.evaluate.goals_experiment import load_goal_experiment
from ligat.viz import goal_training_curves as curves

spec = load_goal_experiment()
pilot_training = curves.training_curves(spec, stage="pilot")
for name, figure in curves.pilot_figures(pilot_training).items():
    show(figure, name)

<div dir="rtl"><h2>4.2 — שלושה קיפולי פיילוט מראים מסלול, ולא הוכחת עדיפות</h2><p>עקומות XGBoost מציגות חציון ו-<a href="#glossary">IQR</a> בין שלושת קיפולי הפיילוט: 1,000 עצים ו-min_child_weight=10, צבע לפי עומק וחלונית לפי קצב למידה. ציר העצים לוגריתמי. מפת האימון מקבעת 1,000 עצים ומציגה חציון לכל עומק וזהירות; n=3 בכל תא, פיזור תיאורי בלבד, ללא מסקנה מובהקת. שאר הצירופים נשמרים בטבלה המלאה.</p><p>ב-CatBoost משווים ברירת מחדל לשלושה עומקי רשת עם קנס 3, חציון ו-IQR על שלוש עונות. loss מקורי של ספרייה אחת אינו בר השוואה לזה של אחרת: CatBoost, GLM ו-Maher עשויים להשמיט log(y!) או לכלול משקל וקנס. אין להסיק איכות חיזוי מהפרש בין הגרפים שלהם. גרף Maher מציג את כל האימונים היומיים: זו מדידת הנדסה, ללא אומדן אוכלוסייה או רווח סמך.</p></div>
<div dir="rtl"><p>איך קוראים: ציר x הוא מספר העצים (לוגריתמי), y הוא loss האימון, צבע הוא עומק וחלונית היא קצב. הקו הוא חציון והפס IQR — החצי האמצעי של התוצאות. בדוגמה 1, 2, 3 החציון הוא 2 והרבעונים הם 1.5 ו-2.5. זה תיאור פיזור בלבד בשלושה קיפולים. עומק גדול וקצב גבוה מקטינים את loss האימון, אך אינם מבטיחים הצלחה בעתיד.</p><p>במפת האימון, שורה היא עומק ועמודה היא זהירות לפני פיצול. צבע ותא מתארים אימון בלבד; יש להשוות לוולידציה בפרק הבא, ולא לבחור לפי הצבע הכהה.</p></div>

<div dir="rtl"><h2>4.3 — טיוטה, לפני QA של Claude ובחירת עמית</h2>
<p><b>אימון מול ולידציה:</b> המפות קוראות את תוצרי full החתומים לאותן תצורות S2-A: 1,000 עצים וקצב 0.1. באימון חציון ו-IQR, ובוולידציה ממוצע משוקלל משחקים ו-CI95 בין קיפולים, n=15 בכל תא. שני הצירים מציגים ערכי רשת בדידים. משקל האימון והרכב העונות שונים מהוולידציה, ולכן הפער אינו אומדן נקי יחיד להתאמת יתר; אבחון E2d משתמש ב-NLL לא משוקלל לשניהם.</p>
<p><b>מפת ההיפר-פרמטרים:</b> אחוז מתרומת Elo הוא 100×(NLL_R0−NLL_candidate)/(NLL_R0−NLL_R1), על אותם 15 קיפולים, עם CI95 בריחוף. 100% פירושו התרומה המלאה של R1 מול R0; ערך שלילי גרוע מ-R0. ברשת המוצגת (1,000 עצים, קצב 0.1), עומק 2 וזהירות 30 נותנים את ה-NLL הנמוך ביותר. זה אינו בוחר תצורה חדשה בסולם.</p>
<p><b>השפעות הפרמטרים:</b> מוצג ממוצע ±<a href="#glossary">SE</a> בין תצורות הרשת, לכל ערך, בנפרד לתכנון A ולתכנון B. n התצורות נמצא בריחוף. זה תיאור של הרשת ולא אפקט סיבתי או דגימות עצמאיות; כש-n=1 ה-SE אינו מוגדר ואינו מוצג. ברירת המחדל נשארת תצורה נפרדת, ולכן נקודה שבה רק היא מיוצגת אינה השוואה מבוקרת.</p>
<p><b>התקדמות הסולם:</b> מוצג הפרש מזווג מול S2, עם ±SE ו-CI95 בריחוף, על n=14 קיפולים עמוקים. השלבים הם תוצאות הסולם המצטבר, ולא ניסוי נפרד בכל משפחה; רק F5 עברה את כלל הכניסה. התוצאות והאבחנות תלויות בהכרעות כלל 6 הפתוחות. אין כאן בחירת מודל ל-M3.</p></div>

In [ ]:
if (spec.output / "full/ladder_report.json").exists():
    frozen_report = curves.frozen_full_report(spec)
    full_training = curves.training_curves(spec, stage="full")
    full_validation = curves.full_validation_table(spec, stage="full")
    for name, figure in curves.full_figures(full_training, full_validation, frozen_report).items():
        show(figure, "E2c_" + name)
else:
    display(HTML('<div dir="rtl">ממתין לדוח D3 חתום; לא מחושבים ציוני מועמדים.</div>'))

<div dir="rtl"><h2>5 — הסולם בודק מידע חדש מול אותו יריב להשוואה</h2>
<p>סולם מוסיף משפחת מידע בכל פעם, ומפריד בין תרומת הפיצ׳ר לבין כוונון המודל. S1 ו-S2 מכווננים ב-A ואחר כך B. בדיקת כניסה משתמשת בתצורת S2 קבועה ובאותם משחקים, עם ובלי המשפחה. במסלול המקוצר דירוג שלבים נוספים מוגבל לחמש תצורות S2, ללא רשת חדשה.</p>
<p>לכל עונה עמוקה מחשבים הפרש מזווג <i>dₛ = L(reference,s) − L(candidate,s)</i>, חיובי משמעו שיפור:
</div>

$$
\bar d={\sum_s n_s d_s\over\sum_s n_s},\quad SE={sd(d_s)\over\sqrt n},\quad CI_{95\%}=\bar d\pm t_{n-1,.975}SE.
$$

<div dir="rtl">
<p>n_s הוא מספר המשחקים בעונה ו-n הוא מספר העונות העמוקות. הממוצע משוקלל במשחקים, ופיזור העונות לחישוב SE אינו משוקלל. כלל 8 בגרסה 2 דורש ממוצע שיפור מעל SE, שיפור בלפחות שלוש מחמש העונות האחרונות, וממוצע שיפור חיובי בחמש האחרונות. רווח הסמך מציג אי-ודאות בין עונות, ולא משנה את שער ההחלטה. למשל, תוספת כושר צריכה לשפר חיזוי של אותן קבוצות באותם משחקים, ולא רק לקבל עונות קלות יותר.</p>
<p>כלל 4 מציע מודל פשוט בתוך SE אחת מהמוביל; כלל 6 מסמן כשהמובילות בעמוקים ובחמש האחרונות שונות; כרגע ההרצה ממשיכה באופן זמני עם הצעת כלל 4, ושומרת open_reviews ותלויות. עמית יכריע אחרי QA. כלל 10 מסמן אי-הסכמה עם <a href="#glossary">RPS</a>. F1-R ו-F1-A הן חלופות; אם שתיהן עוברות מוצעת בעלת השיפור הגדול יותר, עם אפשרות להכרעה מפורשת.</p>
</div>
<div dir="rtl"><p><b>הרצף:</b> S1 (Elo ובית) ← S2 (בסיס) ← בדיקות הוצאה מהבסיס ← משפחה אחת בכל פעם. משפחה שנכנסת מצטרפת לבסיס: לכן F6 נבדקה גם עם F5. במסלול המקוצר בודקים מחדש רק חמש תצורות מובילות מ-S2. כוונון מחדש אינו תרומת המשפחה; תרומה נמדדת באותה תצורה ובאותם משחקים.</p><p><b>כלל 8:</b> d חיובי פירושו שהמשפחה הפחיתה NLL. לדוגמה, ירידה מ-1.42 ל-1.41 נותנת d=0.01. לכל עונה יש d; מחשבים ממוצע משוקלל משחקים ו-SE מפיזור העונות. צריך ממוצע מעל SE, שיפור בשלוש לפחות מחמש האחרונות, וממוצע אחרונות חיובי. טבלת F5 הבאה מפרטת כל עונה וכל תנאי.</p><p><b>כלל 4:</b> בוחרים הצעה פשוטה בתוך SE אחת מהמובילה, לא בהכרח את הממוצע הכי נמוך. למשל פער 0.0003 עם SE של 0.0004 הוא קטן מהאי-ודאות. שבירת תיקו מעבר לעומק ולעצים אינה מוגדרת וממתינה לעמית.</p><p><b>כלל 6:</b> משווים מובילה בעמוקות למובילה בחמש האחרונות (2019/20–2023/24). אם שונות, נשארת הכרעה פתוחה. זה סימון לבדיקה ולא החלטה אוטומטית. כלל 10 מסמן אי-הסכמה בין NLL ל-RPS. RPS מסכם את ריבועי השגיאות בהסתברויות המצטברות של חוץ/תיקו/בית: הטווח לא שלילי ונמוך טוב. למשל תחזית שנותנת לתיקו הסתברות גבוהה נענשת פחות אם אכן היה תיקו. הסימונים והפערים בטבלאות פרק 6.</p></div>
<div dir="rtl"><p>דוגמה ל-RPS עם סדר חוץ/תיקו/בית: תחזית (0.2,0.5,0.3) ותיקו בפועל נותנת ממוצע של 0.2² ו-(0.7−1)², כלומר 0.065. הטווח במימוש הוא 0–1. זו דרך אחרת למדוד כמה התחזית רחוקה מהתוצאה, ולכן ייתכן דגל כלל 10 גם אם NLL מעדיף תצורה אחת.</p></div>

In [ ]:
table(views.ladder_plan_table(), {"stage": "שלב", "features": "עמודות", "meaning": "המידע שנבדק"})
feature_details = display_table(
    qa.feature_guide(),
    {"stage": "???", "field": "?????", "hebrew": "?? ??????", "purpose": "????? ?????"},
    show_index=False,
)._repr_html_()
display(
    HTML(
        "<details><summary>????? ???????? ??????, ???? ?????</summary>"
        + feature_details
        + "</details>"
    )
)

<div dir="rtl"><h2>5.1 — מידע דומה מיוצג פעם אחת</h2>
<p>מסנן <a href="#glossary">Pearson</a> מחושב פעם אחת על פיצ׳רים מספריים ובוליאניים בלבד בעונות הפיתוח. מזהים ו-stage אינם מקבלים מספר סדר מלאכותי. r בטווח [−1,1]; כאן r≈0.957, ולכן שני ההפרשים כמעט מספרים אותו סיפור על העונה הקודמת. שתי ההצעות והכרעת עמית מוצגות בלי לייחס למתאם סיבתיות. ספירת הנתונים היא 8,234 שורות תקפות; זו בדיקת חפיפה תיאורית, לא הוכחת תועלת בחיזוי.</p>
</div>
<div dir="rtl"><p>Pearson r מודד קשר ליניארי, בין −1 ל-1. ערך קרוב ל-1 פירושו ששני פיצ׳רים עולים יחד, ולא ששניהם גורמים לשערים. למשל נקודות והפרש שערים של קבוצה חזקה נעים יחד. עונת החימום הראשונה חסרה עבר, ולכן לזוג נתוני עבר יש פחות שורות שלמות. המסנן מחושב על פיצ׳רים בלבד ב-2006/07–2023/24; שערים קודמים משקפים כוח טוב יותר מנקודות, שכבר מקודדות במיקום. הסרת prev_ppm_diff נעשתה לפני D3.</p></div>

In [ ]:
table(
    qa.correlation_evidence(data),
    {
        "pair": "זוג הפיצ׳רים",
        "r": "Pearson r",
        "n": "שורות שלמות",
        "missing": "שורות ללא עבר מלא",
        "position_r": "מתאם נקודות עם מיקום",
        "scope": "טווח",
        "codex": "הצעת Codex",
        "claude": "הצעת Claude",
        "decision": "הכרעת עמית לפני D3",
    },
    precision={"r": 6, "position_r": 6},
)

<div dir="rtl"><h2>6 — טיוטה, לפני QA של Claude ובחירת עמית</h2><p>הטבלאות קוראות את ladder_report.json החתום. 15 הקיפולים נותנים תמונת פיתוח מלאה; הקיפולים העמוקים משמשים הכרעה; חמש האחרונות בודקות את הליגה העדכנית. סמני כלל 6 ותלויותיהם נשארים פתוחים לעמית. הסרגלים הם שחזור, ואין כאן בחירת מודל ל-M3.</p><p>הצעת S2 היא תצורה פשוטה בקבוצת הקרובות לפי כלל 4, ואינה בהכרח המובילה העמוקה או בחמש האחרונות. ההצעה, התלויות והאבחון כולם זמניים. אין כאן הכרעה חדשה של עמית.</p></div>
<div dir="rtl"><p>הטבלאות הן תיאור של עונות פיתוח שכבר נראו, לפני הכרעות עמית. שיפור מזווג חיובי מול S2 אומר פחות NLL. ל-GLM שמתחיל מעונה 9 יש עונה עמוקה משותפת אחת פחות: ב-2010/11 (עונה 12) היו לו רק שלוש עונות אימון.</p><p>MAE הוא גודל הטעות הממוצעת בשערים; RMSE מעניש במיוחד טעויות גדולות. הם דומים בין המודלים כי רוב הטעות היא רעש המשחק. יתרון בכ-50% משורות קבוצה-משחק הוא בערך הטלת מטבע ברמת השורה, ואינו שקול להימור רווחי או לבחירה נכונה בכל משחק.</p><p>רשת CatBoost התחילה בעומק 4 וב-1,000 עצים, ולכן כיסתה אזור שמועד לשינון בנתונים האלה. הבדיקה בדיעבד מוסיפה תקציב רדוד יותר; היא אינה משנה את הסולם.</p></div>
<div dir="rtl"><p><b>הערת אימות:</b> הטענה המספרית במסמך ה-QA על אחוז מתרומת Elo בחמש האחרונות אינה תואמת לדוח החתום שבטבלה. היא נשארת לבירור (Q61). גם עיגול d̄ של F5 במסמך ה-QA שונה מעיגול הערך החתום (Q60). מוצגים המקורות, ללא שינוי בהחלטה.</p><p>פירוק F5 הוא בדיקת הוצאה מכל צד, לא בחירת פיצ׳ר חדשה. הטבלאות מציגות את ההפרש החתום לכל הוצאה; אם רק הוצאת המוטיבציה של היריבה פוגעת, פירוש אפשרי הוא ירידה בעוצמת ההגנה של יריבה בלי יעד. זו פרשנות תיאורית, לא הוכחת סיבתיות.</p></div>

In [ ]:
report_qa, validation_qa = qa.evidence(spec)
scope_names = {
    "all": "כל קיפולי 2009/10–2023/24",
    "deep": "העונות העמוקות המשותפות",
    "last5": "2019/20–2023/24",
}
for scope in ["all", "deep", "last5"]:
    display(HTML('<div dir="rtl"><b>' + scope_names[scope] + "</b></div>"))
    rows = []
    for name, model in report_qa["readable_metrics"].items():
        s = next(s for s in model["summary"] if s["scope"] == scope)
        rows.append(
            dict(
                model=name,
                label=model["label"],
                nll=s["poisson_nll"],
                probability=s["observed_probability_percent"],
                elo=s["elo_share_percent"],
                mae=s["mae"],
                rmse=s["rmse"],
                better=s["better_non_tie_percent"],
                n=s["rows"],
                status="לא נכנסה; זהה לבסיס אחרי כוונון מחדש"
                if name.startswith("F") and name not in report_qa["entered"]
                else "טיוטה לפני הכרעות עמית",
            )
        )
    table(
        pd.DataFrame(rows),
        {
            "model": "שלב",
            "label": "תצורה",
            "nll": "NLL",
            "probability": "הסתברות גאומטרית %",
            "elo": "אחוז מתרומת Elo",
            "mae": "MAE שערים",
            "rmse": "RMSE שערים",
            "better": "עדיפות בין שורות לא שוות %",
            "n": "שורות",
            "status": "מעמד",
        },
        precision={"nll": 6, "elo": 1, "probability": 1},
    )
paired_rows = []
for name, model in report_qa["readable_metrics"].items():
    s = model["paired"]
    paired_rows.append(
        dict(
            name=name,
            d=s["mean_improvement"],
            se=s["se"],
            lower=s["ci_lower"],
            upper=s["ci_upper"],
            n=s["n"],
            emphasis="סרגל ההשוואה המרכזי מול S2" if name == "EXP-002" else "",
        )
    )
table(
    pd.DataFrame(paired_rows),
    {
        "name": "מודל",
        "d": "שיפור מול S2 — חיובי טוב",
        "se": "SE",
        "lower": "תחתון",
        "upper": "עליון",
        "n": "עונות עמוקות משותפות",
        "emphasis": "הדגשה",
    },
    intervals={"d": ("lower", "upper")},
    precision={"d": 6, "se": 6},
)
entries_qa, f5_seasons = qa.entry_tables(report_qa)
table(f5_seasons, {"season": "עונה", "d": "d של F5", "games": "משחקים"}, precision={"d": 6})
table(
    entries_qa,
    {
        "name": "בדיקה או פירוק F5",
        "d": "שיפור",
        "se": "SE",
        "lower": "תחתון",
        "upper": "עליון",
        "n": "עונות",
        "last5": "שיפור בחמש האחרונות",
        "better": "עדיפות מתוך 5",
        "passes": "כלל 8",
        "entered": "נכנסה",
    },
    intervals={"d": ("lower", "upper")},
    precision={"d": 6, "se": 6, "last5": 6},
)
table(
    qa.review_evidence(report_qa, validation_qa),
    {
        "stage": "שלב",
        "kind": "מועמדת",
        "label": "תצורה",
        "changes": "מה שונה מהעמוקה",
        "gap": "תוספת NLL",
        "se": "SE",
        "n": "עונות",
        "depends": "תלויים",
    },
    precision={"gap": 6, "se": 6},
)
table(
    qa.disagreements(report_qa, validation_qa),
    {"stage": "דגל כלל 10", "score": "מדד", "d": "הפרש מזווג בהצעה", "se": "SE", "n": "עונות"},
    precision={"d": 6, "se": 6},
)

<div dir="rtl"><h2>6.1 — אבחון מאוחר מראה היכן הוולידציה מפסיקה להשתפר</h2><p>אחרי הקפאת ההכרעות אומנו מחדש זוכת S2 ושכנותיה המיידיות בעומק ובקצב, וכן מספר העצים המרבי, ב-15 קיפולים. כל יתר ההגדרות קבועות. בקצה הרשת אין ממציאים שכן מבחוץ; תצורות זהות מוסרות. העקומות מציגות חציון ו-IQR, n=15, של NLL לא משוקלל באימון ובוולידציה; משקלי אופטימיזציית האימון נשארו אלה שנבחרו.</p><p>הקו האנכי מסמן את מספר העצים שנבחר לפני האבחון. כשהוולידציה מתייצבת או עולה בזמן שהאימון ממשיך לרדת, זו עדות אפשרית להתאמת יתר. זה אינו early stopping, אינו בוחר מספר עצים, ואינו משנה שום הכרעה. קובצי האבחון נפרדים, חתומים ומסומנים diagnostic=true.</p><p><b>על הבחירה הזמנית; אם עמית יכריע אחרת, האבחון ירוץ שוב.</b> בהצעה הנוכחית, עומק 2 וקצב 0.05 הם גבולות תחתונים: שכנים מחוץ לרשת דולגו. האבחון כולל ארבע תצורות ו-60 אימונים, והשכנה עם 1,000 עצים נשמרת בנפרד מן ההצעה עם 150 עצים.</p><p>בעקומת 1,000 העצים (עומק 2, קצב 0.05), חציון NLL הוולידציה כמעט מתייצב בין 150 ל-400 עצים (כ-1.418 בשתי הנקודות), ועולה לכ-1.423 ב-1,000; חציון האימון ממשיך לרדת מכ-1.399 ב-150 לכ-1.368 ב-1,000. בקצב 0.1 חציון הוולידציה כמעט זהה ב-100 וב-150 עצים (כ-1.4172). בהצעת 150 העצים עם קצב 0.05 הוולידציה עדיין יורדת בין 100 ל-150. אלה תיאורי חציון ו-IQR של 15 קיפולים, לא הוכחת מובהקות או המלצה לנקודת עצירה; 150 העצים המקוריים והכרעות D3 נשארים כפי שהיו.</p></div>
<div dir="rtl"><p>איך קוראים: קו מלא הוא NLL באימון, מקווקו בוולידציה, והפס הוא החצי האמצעי בין 15 קיפולים. הקו האנכי הוא מספר העצים של ההצעה הזמנית. ״לא משוקלל״ אומר שכל שורת קבוצה-משחק מקבלת משקל זהה בדיווח, גם אם באופטימיזציה יש דעיכה. זו אינה שיטת עצירה מוקדמת. עומק 1 ייבדק בנפרד בפרק 6.2.</p></div>

In [ ]:
diagnostic = curves.load_learning_diagnostics(spec)
show(curves.learning_curve_figure(diagnostic), "E2d_learning_curves_train_vs_validation")

<div dir="rtl"><h2>7 — להבין את המודל, אחרי ההכרעות</h2><p>D3 הושלם. פרק זה יבדוק כיול, יתרון בית שנלמד, SHAP (חלוקת תרומת הפיצ׳רים לחיזוי), λ מול פער Elo, שאריות לפי קבוצה, Maher מול M2 בתחילת עונה ואצל עולות, וכוחות התקפה והגנה. הציפייה שנרשמה מראש היא ש-Elo יהיה הפיצ׳ר החזק ביותר ב-SHAP; זו השערה לבדיקה, לא מסקנה קיימת. הבדיקות על משחקים שנראו מול שלא נראו ייעשו לאחר בחירת המודל, כדי ש-M3 יידע לאיזו רמת רעש לצפות.</p></div>

<div dir="rtl"><h2>8 — הנדסה שומרת על ניסוי עיוור, סיבתי וניתן לחידוש</h2>
<p>pilot שומר λ ו-loss אימון בלבד; אין בו יעד ולידציה או NLL, RPS, log loss או Brier. כל יחידה ויומן חתומים, והאינדקס הוא תצוגה נגזרת. חסר או לא עדכני מוצגים במפורש, בלי דריסה או אימון אוטומטי במחברת. חתימת היחידה מכסה 37 תלויות מפורשות וגרסאות ספריות; קוד תצוגה אינו מבטל יחידות. המאמן אינו כותב בכל איטרציה: היומן נאסף בזיכרון ונכתב פעם אחת בסיום.</p><p>Maher נבדק ב-11,600 אימונים יומיים, ללא כשלים. בטבלאות הבאות זמני מפקד הפיילוט: חציון וממוצע ליחידה, ולא אומדן אוכלוסייה. מדידת עובדים היא הרצה אחת לכל מספר, n=60 יחידות קבועות; אין כאן רווח סמך של תזמון חוזר.</p>
</div>
<div dir="rtl"><p>עיוורון: לפני D3 נשמרו חיזויים והנדסה, ללא ציוני מועמדים. סיבתיות: כשחוזים משחק ב-10.12.2023, Maher רשאי להשתמש בתוצאות עד 9.12 בלבד. חידוש: יחידה חתומה קיימת נטענת במקום אימון חוזר. unit_id הוא שם מזהה של חישוב, manifest הוא רשימת חתימות, ו-Parquet הוא פורמט טבלאי דחוס — אין להם משמעות כדורגלנית.</p><p>CatBoost ברירת המחדל נעשה איטי יותר בפיילוט השני כי הקדימות החדשה הפעילה אותו במקביל ליחידות כבדות; זו תחרות על משאבים, לא שינוי בתוצאה. RSS הוא כמה זיכרון ההרצה תפסה בשיא; שיא D3 לא נמדד. יומן האימון נאסף בזיכרון ונכתב פעם אחת ליחידה.</p></div>

In [ ]:
engineering = views.engineering_tables()
table(
    engineering["backend"],
    {
        "timing_group": "קבוצת תזמון",
        "units": "יחידות",
        "fit_mean": "ממוצע D2 בשניות",
        "fit_median": "חציון D2 בשניות",
        "fit_mean_D2b": "ממוצע D2b בשניות",
        "fit_median_D2b": "חציון D2b בשניות",
        "mean_change_percent": "שינוי בממוצע באחוזים",
    },
)
table(
    engineering["wall"],
    {
        "run": "הרצה",
        "seconds": "זמן CLI בשניות",
        "calibration": "מקדם כיול",
        "rss": "שיא RSS כולל ב-MiB",
    },
)
table(
    engineering["bench"],
    {"n_jobs": "תהליכים", "wall_seconds": "זמן קיר בשניות", "units_per_hour": "יחידות בשעה"},
)

<div dir="rtl"><h3>8.0 — פרטים טכניים ותוצרים שמורים</h3><p>הטבלאות הבאות נועדו לביקורת ולשחזור, ולא לבחירת מודל.</p></div>

In [ ]:
import json
from pathlib import Path

execution = json.loads(Path("data/reports/T-014/D3_execution.json").read_bytes())
table(
    pd.DataFrame(
        [
            dict(
                seconds=execution["wall_seconds"],
                units=len(validation_qa),
                jobs=spec.n_jobs,
                minimum=qa.outcome_diagnostics(validation_qa).loss.min(),
            )
        ]
    ),
    {"seconds": "שניות D3", "units": "יחידות", "jobs": "תהליכים", "minimum": "מינימום H/D/A"},
)

In [ ]:
prior = views.frozen_prior()
table(
    pd.DataFrame([prior]),
    {
        "a_star": "a* — התקפה יחסית בלוג",
        "b_star": "b* — ספיגה יחסית בלוג",
        "source_sha256": "חתימת מקור",
        "team_seasons": "קבוצות־עונות",
    },
    precision={"a_star": 6, "b_star": 6},
)

In [ ]:
status = views.pilot_status()
table(
    status.groupby("status").size().rename("units").reset_index(),
    {"status": "מצב C5", "units": "יחידות"},
)
table(
    pd.DataFrame(
        [
            dict(
                stage=stage,
                rows=len(curves.training_curves(spec, stage=stage)),
                bytes=(spec.output / stage / "training_curves.parquet").stat().st_size,
            )
            for stage in ["pilot", "full"]
        ]
    ),
    {"stage": "שלב", "rows": "שורות עקומות", "bytes": "גודל בבתים"},
)

In [ ]:
table(
    pd.DataFrame(
        [
            {
                key: diagnostic[key]
                for key in (
                    "configs",
                    "folds",
                    "files",
                    "frozen_artifacts",
                    "frozen_artifacts_unchanged",
                )
            }
        ]
    ),
    {
        "configs": "תצורות אבחון",
        "folds": "קיפולים",
        "files": "קבצים",
        "frozen_artifacts": "מקורות",
        "frozen_artifacts_unchanged": "נשמרו",
    },
)

<div dir="rtl"><h3>8.0.1 — נפח יומנים ותוצאות בדיקות ההנדסה</h3><p>היומנים נכתבים פעם אחת בסוף כל יחידה. נפח הוא מדידה הנדסית, ללא משמעות כדורגלנית. טבלאות החתימות וההיסטוריה נועדו לשחזור בלבד.</p></div>

In [ ]:
journal_sizes = pd.DataFrame(
    [
        dict(
            stage=stage,
            bytes=sum(p.stat().st_size for p in (spec.output / stage).glob("*.trainlog.json.gz")),
            journals=len(list((spec.output / stage).glob("*.trainlog.json.gz"))),
        )
        for stage in ["pilot", "full"]
    ]
)
table(journal_sizes, {"stage": "שלב", "bytes": "בתים דחוסים", "journals": "יומנים"})
outcome_table = qa.outcome_diagnostics(validation_qa)
table(
    pd.DataFrame(
        [
            dict(
                configurations=len(outcome_table),
                minimum=outcome_table.loss.min(),
                flags=int(outcome_table.flag.sum()),
            )
        ]
    ),
    {"configurations": "תצורות", "minimum": "מינימום log loss של H/D/A", "flags": "דגלים מתחת ל-1"},
)

<div dir="rtl"><h2>8.1 — ציון טוב מדי דורש בדיקת דליפה</h2>
<p>log loss של בית/תיקו/חוץ נגזר ממטריצת 6×6: שערים 0–4 ו-5+, ותא שבו שתי הקבוצות מבקיעות חמש ומעלה נחשב תיקו לפי הקירוב של חמש ומעלה. הטווח הוא [0,∞), ופחות טוב יותר. ממוצע תצורה מתחת ל-0.98 עוצר כחשד לדליפה; קיפול בודד רק מסומן. כל תצורה מתחת ל-1.00 מסומנת ל-QA. שלוש התוצאות מסודרות בית/תיקו/חוץ; RPS בטווח [0,1] מתחשב גם בסדר הזה. ״משחקים שנראו מול שלא נראו״ ממתין לבדיקה לאחר בחירת המודל.</p>
</div>
<div dir="rtl"><p>כששתי הקבוצות מבקיעות 5 ומעלה, התא המשותף נספר כתיקו בקירוב מטריצת התוצאות. הסף 0.98 הוא עצירה לחשד לדליפה, ו-1.00 הוא דגל לבדיקה. למשל הצצה לתוצאה ב-Elo הופכת חיזוי לקל באופן מלאכותי; זו הדגמת דליפה ולא מודל מועמד. נתוני האימות בפועל מופיעים בטבלה הבאה.</p></div>

<div dir="rtl"><h2>9 — M3 יקבל את מדיניות האימון שנבדקה בפועל</h2>
<p>עמית בחר as_evaluated: אותו unit_id ואותה היסטוריה כמו בהערכה. למשל, Maher יומי משתמש גם בתוצאות מוקדמות בעונה הנוכחית, בתנאי train_end_date &lt; as_of. המדיניות החלופית prior_seasons_only מחריגה את העונה הנוכחית ומחייבת יחידות אחרות. אין ברירת מחדל; המדיניות היא חלק מחתימת השכבה. train_end_season מתאר עונה שלמה, ולכן תמיד קטן מעונת היעד, גם כשהתאריך שייך לעונה הנוכחית.</p><p>היצוא במצב as_evaluated קורא יחידות full שהושלמו, בלי לאמן חסרות ובלי לשנות את יחידות D3. Parquet ו-manifest נחתמים עם קלט, קוד, תצורה ופלט. חיבור שתי שורות המשחק נותן lambda_home ו-lambda_away. השכבה האמיתית ממתינה ל-QA ולבחירת עמית.</p>
</div>
<div dir="rtl"><p>דוגמת חוזה בלבד למשחק בדצמבר 2023: Maher יכול להתאמן עד 9.12.2023 ולכלול את העונה הנוכחית, אך העונה השלמה האחרונה היא 2022/23. XGBoost לקיפול משתמש רק בעונות שלמות עד 2022/23, ולכן אינו מסומן כמאומן על העונה הנוכחית. לא נכתבה כאן שכבת λ אמיתית; הפרטים הטכניים בפרק 8.</p></div>

In [ ]:
schema = views.lambda_schema_table()
schema["example"] = schema.column.map(
    {
        "game_id": "משחק המחשה",
        "team_id": "קבוצת המחשה",
        "season_id": "2023/24 (25)",
        "as_of": "10.12.2023",
        "lambda": "1.6 — המחשה",
        "train_end_season": "2022/23",
        "train_end_date": "9.12.2023 ב-Maher",
        "trained_on_current_season": "כן ב-Maher; לא ב-XGBoost",
        "config_id": "חתימת התצורה",
        "prematch_version": "prematch_v2",
    }
)
table(schema, {"column": "עמודה", "meaning": "משמעות וחוזה", "example": "דוגמה להמחשה בלבד"})
illustration = pd.DataFrame(
    [
        dict(
            model="Maher",
            as_of="10.12.2023",
            train_end_date="9.12.2023",
            trained_on_current_season=True,
            train_end_season="2022/23",
        ),
        dict(
            model="XGBoost",
            as_of="10.12.2023",
            train_end_date="סוף עונת 2022/23",
            trained_on_current_season=False,
            train_end_season="2022/23",
        ),
    ]
)
table(
    illustration,
    {
        "model": "מודל — המחשת חוזה בלבד",
        "as_of": "בוקר החיזוי",
        "train_end_date": "גבול האימון",
        "trained_on_current_season": "העונה הנוכחית נכללה",
        "train_end_season": "עונה שלמה אחרונה",
    },
)

<div dir="rtl" id="glossary"><h2>10 — מילון</h2><table><thead><tr><th>מושג</th><th>מה זה</th><th>דוגמה</th><th>משמעות בכדורגל</th></tr></thead><tbody><tr><td>Elo</td><td>כוח יחסי המתעדכן מתוצאות עבר</td><td>פער 100 נקודות</td><td>פערי כוח בין יריבות</td></tr><tr><td>GLM</td><td>מודל לוג-ליניארי לקצב שערים</td><td>ביתיות פי 1.2 — המחשה</td><td>השפעה קבועה על קצב</td></tr><tr><td>XGBoost</td><td>עצים מתקנים זה את זה</td><td>עץ עומק 2 עם ארבעה קצוות</td><td>שילובי מצבים</td></tr><tr><td>CatBoost</td><td>עצים עם טיפול בקטגוריות</td><td>שלב הליגה הוא קטגוריה</td><td>השוואת שיטה</td></tr><tr><td>Maher</td><td>כוח התקפה וספיגה לכל קבוצה</td><td>התקפה פי 1.1 — המחשה</td><td>כוח קבוצה נפרד לכל צד</td></tr><tr><td>d̄</td><td>ממוצע משוקלל של הפרשי NLL מזווגים</td><td>1.42−1.41=0.01</td><td>חיובי פירושו חיזוי טוב יותר</td></tr><tr><td>אחוז מתרומת Elo</td><td>100 כפול השיפור מעל R0 חלקי שיפור R1 מעל R0</td><td>100% שווה לסרגל Elo</td><td>כמה מהמידע על הכוח הוחזר</td></tr><tr><td>הסתברות גאומטרית</td><td>exp של מינוס NLL הממוצע</td><td>NLL=1.41 נותן כ-24.4%</td><td>משקל לתוצאות שבאמת קרו</td></tr><tr><td>RPS</td><td>שגיאה ריבועית של הסתברויות מצטברות</td><td>0 הוא חיזוי מושלם</td><td>דירוג תחזית בית/תיקו/חוץ</td></tr><tr><td>log loss</td><td>מינוס לוג ההסתברות לתוצאה שנצפתה</td><td>הסתברות חצי נותנת כ-0.693</td><td>קנס על ביטחון מוטעה</td></tr><tr><td>Pearson r</td><td>קשר ליניארי בין −1 ל-1</td><td>r קרוב ל-1 מצביע על תנועה יחד</td><td>נקודות והפרש שערים של קבוצה חזקה</td></tr><tr><td>פיזור יתר</td><td>שונות גדולה מהממוצע במודל ספירה</td><td>ממוצע 1 ושונות 2 — המחשה</td><td>יותר משחקי קצה מ-Poisson</td></tr><tr><td>IQR</td><td>החצי האמצעי של התוצאות</td><td>1,2,3 נותנים רבעונים 1.5 ו-2.5</td><td>פיזור בין עונות</td></tr><tr><td>SE</td><td>שגיאת תקן של ממוצע בין עונות</td><td>פער קטן מ-SE אינו מובחן ברעש</td><td>יציבות בין עונות</td></tr><tr><td>כלל 4</td><td>הצעה פשוטה בתוך SE מהמובילה</td><td>עומק קטן קודם</td><td>שבירת תיקו נשארת לעמית</td></tr><tr><td>כלל 6</td><td>מובילות עמוקות ואחרונות שונות</td><td>שתי תצורות זמן מחצית שונות</td><td>בדיקת שינוי בליגה</td></tr><tr><td>כלל 8</td><td>שער כניסת מידע עם שלושה תנאים</td><td>F5 בטבלה המחושבת</td><td>האם מידע חדש מוסיף</td></tr><tr><td>כלל 10</td><td>אי-הסכמה בין NLL ל-RPS</td><td>שלושה דגלים בדוח D3</td><td>רגישות לדרך מדידת איכות</td></tr><tr><td>walk-forward</td><td>אימון בעבר ובדיקה בעונה הבאה</td><td>2006/07–2008/09 לפני 2009/10</td><td>מדמה חיזוי בזמן אמת</td></tr><tr><td>early stopping</td><td>עצירה לפי עונת ולידציה</td><td>אינה מופעלת בניסוי הזה</td><td>מונעת בחירת עצים בדיעבד</td></tr><tr><td>שלבים A ו-B</td><td>מבנה העץ ואחריו ריסון ודעיכה</td><td>עומק קודם, זמן מחצית אחר כך</td><td>אין משמעות כדורגלנית ישירה</td></tr><tr><td>המסלול המקוצר</td><td>שימוש בחמש תצורות S2 בהמשך</td><td>אותן תצורות למשפחה חדשה</td><td>תקציב חיפוש מוגבל מראש</td></tr><tr><td>הכרעה פתוחה</td><td>בחירה שדורשת החלטת עמית</td><td>מובילות שונות בכלל 6</td><td>אינה החלטה אוטומטית</td></tr><tr><td>as_of</td><td>גבול המידע בבוקר המשחק</td><td>משחק ב-10.12 משתמש בעבר בלבד</td><td>מונע הצצה לעתיד</td></tr><tr><td>D3</td><td>הרצת הסולם המלאה</td><td>תוצאות חתומות בפרק 6</td><td>ניסוי הפיתוח</td></tr><tr><td>Γ ו-deviance</td><td>Γ מכלילה עצרת; deviance מסיר קבוע של היעד</td><td>Γ(y+1)=y!; אין להשוות deviance ל-NLL גולמי</td><td>כלי חישוב, לא כוח קבוצה</td></tr><tr><td>λ</td><td>ממוצע שערים חיובי</td><td>1.6 שערים צפויים</td><td>אינו תוצאה מובטחת</td></tr><tr><td>NLL</td><td>מינוס לוג ההסתברות למה שקרה</td><td>−ln(0.323)≈1.130</td><td>הפתעה נמוכה פירושה חיזוי טוב</td></tr><tr><td>Γ</td><td>הכללה של עצרת</td><td>Γ(3)=2!</td><td>קבוע חישובי של מספר השערים</td></tr><tr><td>deviance</td><td>הפסד Poisson שמחסיר קבוע תלוי יעד</td><td>2[y ln(y/λ)−(y−λ)]</td><td>אין להשוות ישירות ל-NLL גולמי</td></tr><tr><td>RSS</td><td>כמה זיכרון ההרצה תפסה בשיא</td><td>נמדד בפיילוט, לא נמדד ב-D3</td><td>אין משמעות כדורגלנית</td></tr><tr><td>QA</td><td>בדיקת איכות עצמאית</td><td>Claude בודק קוד ומספרים</td><td>מונע הכרעות על טעות תצוגה</td></tr><tr><td>unit_id, manifest, Parquet</td><td>מזהה חישוב, רשימת חתימות וטבלה דחוסה</td><td>יחידה קיימת נטענת בלי אימון</td><td>אין משמעות כדורגלנית</td></tr></tbody></table></div>